# <font color="purple">EDA</font>
## <font color="purple">(exploratory data analysis)</font>

# <font color="1E93AB">Import Libraries</font>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.linear_model import LinearRegression

# <font color="1E93AB">Load Dataset</font>

In [ ]:
part1_df =pd.read_excel('OnlineRetail_part1.xlsx')
part2_df =pd.read_csv('OnlineRetail_part2.csv')
part1_df

In [ ]:
part2_df

# <font color="1E93AB">Merge Datasets</font>

In [ ]:
df =pd.concat([part1_df,part2_df])
df

# <font color="1E93AB">Explore Data</font>

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
#check duplicated rows
df.duplicated().sum()

In [ ]:
# remove duplicated rows
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df

# <font color="1E93AB">Clean Data</font>

## - first column

In [ ]:
df['InvoiceNo'].isna().sum()

In [ ]:
invalid_numbers = []
for i in df['InvoiceNo']:
    if str(i).isnumeric():
        pass
    else:
        invalid_numbers.append(i)

In [ ]:
#clean the invoiceno column
df['InvoiceNo']=df['InvoiceNo'].astype(str).str.strip()
# #separate cancellations 
# returns=df[df['InvoiceNo'].str.startswith('C')]
# df =df[~df['InvoiceNo'].str.startswith('C')]

## - second column

In [ ]:
df

In [ ]:
df['StockCode'].value_counts()

In [ ]:
# clean the StockCode
df['StockCode'] = df['StockCode'].astype(str).str.strip()

# find irrelevant values
irrelevant_values = [i for i in df['StockCode'].unique() if i.isalpha()]

# remove irrelevant values
df.drop(df[df['StockCode'].isin(irrelevant_values)].index, inplace=True)

# check the process
irrelevant_values = [i for i in df['StockCode'].unique() if i.isalpha()]
irrelevant_values

## - third column

In [ ]:
df['Description'].value_counts()

In [ ]:
df['Description'].isna().sum()

In [ ]:
df =df.dropna(subset=['Description'])

In [ ]:
df['Description'].isna().sum()

In [ ]:
#convert to str 
df['Description'] =df['Description'].astype(str).str.strip().str.upper()
irrelevant_words =['TEST','SAMPLES','DAMAGED','UNKNOWN','CHECK','ADJUST','?? missing','???MISSING','MISSING?']
#replace irrelevant words
mode_df =df['Description'].mode()
for word in irrelevant_words:
    df['Description'].replace(word,'WHITE HANGING HEART T-LIGHT HOLDER',inplace=True)

In [ ]:
df['Description'].value_counts()

## - fourth column

In [ ]:
df['Quantity'].isna().sum()

In [ ]:
df['Quantity'].value_counts()

In [ ]:
#less than one values
irr_quantity =[]
for i in df['Quantity']:
    if 1>i:
        irr_quantity.append(i)
for i in irr_quantity:
    df['Quantity'].replace(i,1.0,inplace=True)


In [ ]:
df['Quantity'].value_counts()

In [ ]:
df['Quantity'].fillna(1.0,inplace=True)

In [ ]:
df['Quantity'].isna().sum()

In [ ]:
df['Quantity']=df['Quantity'].astype(int)

## - fifth column

In [ ]:
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])

## - sixth column

In [ ]:
df['UnitPrice'].info()

In [ ]:
df[df['UnitPrice']<=0]

In [ ]:
most_common_price = df['UnitPrice'].mode()[0]
most_common_price

In [ ]:
# positive values
positive_prices = df[df['UnitPrice'] > 0]['UnitPrice']

# mode of data
most_common_positive_price = positive_prices.mode()[0]

# replace 0 with mode
df.loc[df['UnitPrice'] <= 0, 'UnitPrice'] = most_common_positive_price
df['UnitPrice'].fillna(most_common_positive_price,inplace=True)

## - seventh column

In [ ]:
invalid_ids = []

for id in df['CustomerID']:
    try:
        float(id)  
    except (ValueError, TypeError):
        invalid_ids.append(id)

In [ ]:
df['CustomerID'] =df['CustomerID'].fillna('unknown')

In [ ]:
df['CustomerID'].dropna().isna().sum()

In [ ]:
lst_ = []

for i in df['CustomerID'].unique():
    try:
        float(i)
    except (ValueError, TypeError):
        print(i)
        lst_.append(i)

## - eight column

In [ ]:
df['Country'].unique()

In [ ]:
df['Country'].mode()

In [ ]:
df.loc[(df['Country'] == ' ') | (df['Country'] == 'Unspecified'), 'Country'] = 'United States'

In [ ]:
df['Country'].unique()

In [ ]:
df.to_csv('clean_OnlineRetail.csv',index=False)

# <font color="320A6B">Descriptive Analysis</font>

### <font color="0F828C">- average price</font>
### <font color="0F828C">- minimum price</font>
### <font color="0F828C">- maximum price</font>

In [ ]:
average_price =df['UnitPrice'].mean()
min_price =df['UnitPrice'].min()
max_price =df['UnitPrice'].max()
print(f'Average price: {average_price :.4f}\nMinimum price: {min_price :.4f}\nMaximum price: {max_price :.4f}')

### <font color="0F828C">- the highest total sale</font>

In [ ]:
max_quantity =df['Quantity'].max()
top_products =df[df['Quantity']==max_quantity]
df_top_products =top_products[['Quantity','Description']]
df_top_products

### <font color="0F828C">- transactions per country</font>

In [ ]:
# number of transactions per country
country_transactions = df.groupby('Country')['InvoiceNo'].nunique()

# print the result
for country, num in country_transactions.items():
    print(f'{country} : {num} transactions')

### <font color="0F828C">- distribution of transactions per country</font>

In [ ]:
country_transactions = df.groupby('Country')['InvoiceNo'].nunique().sort_values(ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(country_transactions.index, country_transactions.values, color='skyblue')
plt.xlabel('Number of Transactions')
plt.title('Transaction Distribution by Country')
plt.tight_layout()
plt.grid()
plt.show()

# <font color="696FC7">Diagnostic Analysis</font>

### <font color="4FB7B3">- 1)Why some customers make more purchases</font>

In [ ]:
df.columns

In [ ]:
# transactions per country
country_transactions = df.groupby('Country')['InvoiceNo'].nunique()
top_countries = country_transactions.sort_values(ascending=False).head(10)

# transactions for top  stockcodes 
stock_code_transactions = df.groupby('StockCode')['InvoiceNo'].nunique()
top_stock_codes = stock_code_transactions.sort_values(ascending=False).head(10)

In [ ]:
# subplot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# barchart1
axes[0].barh(top_countries.index[::-1], top_countries.values[::-1], color='lightgreen')
axes[0].set_title('Top 10 Countries with Most Transactions')
axes[0].set_xlabel('Number of Transactions')
axes[0].set_ylabel('Country')

# barchart2
axes[1].barh(top_stock_codes.index[::-1], top_stock_codes.values[::-1], color='skyblue')
axes[1].set_title('Top 10 StockCodes with Most Transactions')
axes[1].set_xlabel('Number of Transactions')
axes[1].set_ylabel('StockCode')

# space between them
plt.tight_layout()
plt.show()

### <font color="4FB7B3">- 2)Do returns occure for specific products for?</font>

In [ ]:
# seperate returns
returns_df = df[df['InvoiceNo'].astype(str).str.startswith('C')]
# normal sales
sales_df = df[~df['InvoiceNo'].astype(str).str.startswith('C')] 

# number of returns per stockcode
returns_per_product = returns_df.groupby('StockCode')['Quantity'].count().sort_values(ascending=False)

# number of sales
sales_per_product = sales_df.groupby('StockCode')['Quantity'].count().sort_values(ascending=False)

# the return ratio to total sales
return_ratio = (returns_per_product / (sales_per_product + 1)) * 100 

# top returns
top_returns = returns_per_product.head(10)


#### bar chart

In [ ]:
# bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# number of returns
sns.barplot(x=top_returns.values, y=top_returns.index, ax=axes[0], palette='mako_r')
axes[0].set_title('Top 10 Products with Most Returns')
axes[0].set_xlabel('Number of Returns')
axes[0].set_ylabel('StockCode')

# percentage of returns
sns.barplot(x=top_return_ratio.values, y=top_return_ratio.index, ax=axes[1], palette='coolwarm')
axes[1].set_title('Top 10 Products with Highest Return Ratio (%)')
axes[1].set_xlabel('Return Ratio (%)')
axes[1].set_ylabel('StockCode')

# seperation of them
plt.tight_layout()
plt.show()

### <font color="4FB7B3">- 3)relation between priceunit and quantity</font>

In [ ]:
# correlation between unitprice & quantity
correlation = df['Quantity'].corr(df['UnitPrice'])
print("Correlation between UnitPrice and Quantity:", correlation)

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(x='UnitPrice', y='Quantity', data=df)
plt.title('Relationship between Unit Price and Quantity Sold')
plt.xlabel('Unit Price')
plt.ylabel('Quantity Sold')
plt.show()

# <font color="696FC7">Predictive Analysis</font>

### <font color="58A0C8">- 1)Predict next month's sales per product</font>

In [ ]:

# فرض میکنیم df شامل ستون‌های 'InvoiceDate', 'StockCode', 'Quantity' است
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# اضافه کردن ستون ماه
df['Month'] = df['InvoiceDate'].dt.month
df['Year'] = df['InvoiceDate'].dt.year

# جمع فروش ماهانه برای هر محصول
monthly_sales = df.groupby(['StockCode', 'Year', 'Month'])['Quantity'].sum().reset_index()

# ویژگی‌ها (Features)
monthly_sales['PrevMonthSales'] = monthly_sales.groupby('StockCode')['Quantity'].shift(1)  # فروش ماه قبل
monthly_sales['Prev2MonthSales'] = monthly_sales.groupby('StockCode')['Quantity'].shift(2)  # فروش 2 ماه قبل

# ویژگی‌ها و برچسب‌ها (Label)
monthly_sales = monthly_sales.dropna(subset=['PrevMonthSales', 'Prev2MonthSales'])  # حذف داده‌های ناقص

# ویژگی‌ها: ماه، فروش ماه قبل و فروش 2 ماه قبل
X = monthly_sales[['Month', 'PrevMonthSales', 'Prev2MonthSales']]

# برچسب: فروش
y = monthly_sales['Quantity']

# مدل خطی
model = LinearRegression()
model.fit(X, y)

# پیش‌بینی فروش برای ماه آینده
# برای پیش‌بینی ماه آینده، باید ماه جدید و فروش ماه‌های قبلی را داشته باشیم.
last_row = monthly_sales.iloc[-1]
next_month = last_row['Month'] % 12 + 1  # ماه بعد
prev_month_sales = last_row['Quantity']  # فروش ماه قبل
prev2_month_sales = monthly_sales[monthly_sales['Month'] == (last_row['Month'] - 1) % 12 + 1]['Quantity'].iloc[-1]  # فروش 2 ماه قبل

# ویژگی‌های جدید برای پیش‌بینی ماه آینده
X_new = np.array([[next_month, prev_month_sales, prev2_month_sales]])

# پیش‌بینی فروش ماه آینده
pred = model.predict(X_new)[0]

# نمایش پیش‌بینی
print(f"پیش‌بینی فروش ماه آینده برای محصول {last_row['StockCode']}: {pred:.0f} واحد")